In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent          # ein Ordner hoch
SRC = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC))

In [ ]:
from context.deepthought_context import DeepthoughtExecContext
from db.data_preperation import extract_text, embed_chunks
from db.db_procedures import insert_chunks, query_similar_chunks, get_db_credentials, drop_table, create_chunk_table

ctx = DeepthoughtExecContext()

In [ ]:
chunks = extract_text(
    path="../src/db/documents/Operating_Systems.pdf",
    doc_title="Operating_Systems"
)

chunks = embed_chunks(ctx=ctx, chunks=chunks)

In [ ]:
chunks

In [ ]:

host, port, db, user, password = get_db_credentials()

conn = ctx.establish_db_connection(host=host, port=port, dbname=db, user=user, password=password)
#cleanup db
drop_table(conn=conn)
create_chunk_table(conn=conn)
insert_chunks(conn=conn, chunks=chunks)

In [ ]:
query_emb = ctx.create_multiple_embeddings(["What do you know about threads?"])[0]

results = query_similar_chunks(conn, query_emb, top_k=5)

for r in results:
    chunk_id = r[0]
    title = r[1]
    page = r[2]
    cid = r[3]
    text = r[4]
    similarity = r[5]

    print("--------------------------------------------------")
    print(f"ID: {chunk_id}")
    print(f"Dokument: {title}")
    print(f"Seite: {page}")
    print(f"Chunk: {cid}")
    print(f"Similarity: {similarity:.4f}")
    print(f"Text:\n{text}")
    print("--------------------------------------------------\n")